# تنظيف بيانات عقود الإيجار
**المدخل:** `rental clauses.xlsx`  
**المخرج النهائي:** `rental_contracts_master_clean.xlsx`

| الخطوة | الوصف |
|--------|-------|
| Step 1 | قراءة الملف الأصلي + تنظيف أولي |
| Step 2 | تنظيف القيم + كشف المكررات |
| Step 3 | تحقق من الجودة |
| Step 4 | حفظ الملف الأساسي |
| Step 5 | توحيد المراجع عبر `reference_update.py` |
| Step 6 | تطبيق الـ keywords |
| Step 7 | تحقق نهائي |

In [1]:
import pandas as pd

# ── Step 1: قراءة الملف الأصلي وتنظيف أولي ──
df = pd.read_excel("rental clauses.xlsx")

# توحيد أسماء الأعمدة
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# تنظيف النصوص
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

print(f"عدد الصفوف: {len(df)}")
print(f"الأعمدة: {list(df.columns)}")
df.head()

عدد الصفوف: 76
الأعمدة: ['field', 'title_ar', 'category', 'contract_subtype', 'required', 'risk_if_missing', 'description', 'keywords', 'reference']


,field,title_ar,category,contract_subtype,required,risk_if_missing,description,keywords,reference
0,contract_duration,مدة العقد,contract_terms,both,نعم,غموض في تحديد انتهاء العقد والتجديد,تحديد مدة الإيجار بدقة (شهور أو سنوات) وتاريخ ...,مدة، سنة، شهر، انتهاء,البند ثامناً
1,contract_start_date,تاريخ بدء العقد,contract_terms,both,نعم,خلاف حول بداية احتساب المدة والاستحقاقات,تحديد التاريخ الفعلي لبدء سريان عقد الإيجار,تاريخ البداية، بدء العقد,شرط اتفاقي — يحدده الطرفان
2,automatic_renewal,التجديد التلقائي,contract_terms,both,نعم,استمرار العقد أو إنهاؤه دون علم أحد الطرفين,يتجدد عقد الإيجار تلقائياً ما لم يُشعر أحد الط...,تجديد تلقائي، 60 يومًا، إشعار,البند ثامناً
3,renewal_notice_period,مهلة إشعار عدم التجديد,contract_terms,both,نعم,إنهاء مفاجئ أو التزام بتجديد غير مرغوب,يجب إشعار الطرف الآخر بعدم الرغبة في التجديد ق...,إشعار، 60 يومًا، عدم تجديد,البند ثامناً
4,renewal_notice_personal_use,مهلة الإشعار للاستخدام الشخصي (365 يومًا),contract_terms,residential,نعم,إخلاء مفاجئ للمستأجر بدون مهلة كافية,إذا أراد المؤجر إخلاء العقار لاستخدامه الشخصي ...,إشعار، 365 يومًا، استخدام شخصي، أقارب,قرار زيادة مدة الإشعار


In [2]:
# ── Step 2: تنظيف القيم وكشف المكررات ──
df["required"]         = df["required"].astype(str).str.strip()
df["category"]         = df["category"].astype(str).str.strip().str.lower()
df["field"]            = df["field"].astype(str).str.strip().str.lower()
df["contract_subtype"] = df["contract_subtype"].astype(str).str.strip().str.lower()

print("required:")
print(df["required"].value_counts())

print("\ncategory:")
print(df["category"].value_counts())

print("\ncontract_subtype:")
print(df["contract_subtype"].value_counts())

dups = df["field"].value_counts()
dups = dups[dups > 1]
print("\nمكررات:")
print(dups if len(dups) > 0 else "لا يوجد مكررات")

required:
required
نعم    58
لا     18
Name: count, dtype: int64

category:
category
obligations           23
compensation          21
termination_exit      16
contract_terms         7
legal_compliance       7
general_provisions     2
Name: count, dtype: int64

contract_subtype:
contract_subtype
both           74
residential     2
Name: count, dtype: int64

مكررات:
لا يوجد مكررات


In [3]:
# ── Step 3: تحقق من الجودة ──
print("nulls:")
print(df.isnull().sum())

print("\nقيم فارغة:")
print((df == "").sum())

print(f"\nإجمالي البنود: {len(df)}")
print(f"required نعم:              {(df['required'] == 'نعم').sum()}")
print(f"required لا:               {(df['required'] == 'لا').sum()}")
print(f"contract_subtype both:        {(df['contract_subtype'] == 'both').sum()}")
print(f"contract_subtype residential: {(df['contract_subtype'] == 'residential').sum()}")

nulls:
field               0
title_ar            0
category            0
contract_subtype    0
required            0
risk_if_missing     0
description         0
keywords            0
reference           0
dtype: int64

قيم فارغة:
field               0
title_ar            0
category            0
contract_subtype    0
required            0
risk_if_missing     0
description         0
keywords            0
reference           0
dtype: int64

إجمالي البنود: 76
required نعم:              58
required لا:               18
contract_subtype both:        74
contract_subtype residential: 2


In [4]:
# ── Step 4: حفظ الملف الأساسي قبل المراجع والـ keywords ──
df.to_excel("rental_contracts_master_clean.xlsx", index=False)
print("تم الحفظ الأولي ✅")

تم الحفظ الأولي ✅


In [ ]:
# ── Step 5: توحيد المراجع ──
# reference_update.py يقرأ rental_contracts_master_clean.xlsx
# ويطبق المراجع الموحدة على كل البنود الـ 76 ثم يحفظ
%run reference_update.py

In [5]:
# ── Step 6: تطبيق الـ keywords (عربي + إنجليزي) ──
df = pd.read_excel("rental_contracts_master_clean.xlsx")

keywords_map = {
    'contract_duration':                  'مدة العقد، سنة، شهر، انتهاء، مدة الإيجار، duration، lease term، contract period، rental period',
    'contract_start_date':                'تاريخ البداية، بدء العقد، تاريخ المباشرة، start date، commencement date، lease start',
    'automatic_renewal':                  'تجديد تلقائي، 60 يومًا، إشعار، تجديد العقد، automatic renewal، lease renewal، renew automatically',
    'renewal_notice_period':              'إشعار عدم التجديد، 60 يومًا، عدم التجديد، notice period، non-renewal notice، end of lease',
    'renewal_notice_personal_use':        'إشعار 365 يوم، استخدام شخصي، أقارب، إخلاء، personal use، one year notice، owner occupancy',
    'ejar_registration':                  'تسجيل العقد، منصة إيجار، توثيق العقد، ejar، registration، ejar platform، contract registration',
    'total_rent':                         'أجرة إجمالية، قيمة الإيجار، مبلغ الإيجار، الأجرة، total rent، rent amount، monthly rent، annual rent',
    'rent_payment_schedule':              'موعد الدفع، شهري، سنوي، تحويل بنكي، دفع الإيجار، payment schedule، due date، rent payment',
    'rent_increase_prohibition':          'زيادة إيجار، حظر زيادة، تثبيت الإيجار، رفع الإيجار، rent increase، freeze، no rent increase',
    'vacant_property_rent_cap':           'عقار شاغر، سقف الأجرة، آخر عقد، حد الإيجار، vacant property، rent cap، price ceiling',
    'deposit_security':                   'ضمان، تأمين، مبلغ الضمان، استرداد التأمين، security deposit، deposit، refundable deposit',
    'deposit_dispute_resolution':         'نزاع الضمان، استرداد الضمان، خبراء، محكمة، deposit dispute، deposit refund',
    'earnest_money':                      'عربون، مقدم ثمن، 5%، عربون العقد، earnest money، down payment، booking fee',
    'non_payment_eviction':               'عدم سداد، إخلاء، تأخر دفع، طرد المستأجر، eviction، non-payment، rent arrears',
    'structural_defect_eviction':         'عيوب هيكلية، تقرير فني، إخلاء، عيوب البناء، structural defect، unsafe building، eviction',
    'personal_use_eviction':              'استخدام شخصي، أقارب الدرجة الأولى، 365 يومًا، personal use eviction، owner use، family use',
    'mutual_termination':                 'إنهاء بالتراضي، اتفاق الطرفين، فسخ ودي، mutual termination، termination by consent',
    'alterations_permission':             'تعديلات، إذن، إعادة الحال، تغيير العقار، تحسينات، alterations، modifications، restore',
    'subletting_prohibition':             'تأجير من الباطن، تنازل، موافقة خطية، إيجار من الباطن، subletting، sublease، sublet',
    'damage_liability':                   'أضرار، مسؤولية، تعويض، إصلاح الأضرار، damage، liability، compensation، property damage',
    'riyadh_rent_freeze':                 'الرياض، تجميد إيجار، 5 سنوات، حظر زيادة، Riyadh rent freeze، rent control، تثبيت الرياض',
    'whistleblower_reward':               'مكافأة إبلاغ، 20%، مخالفة، البلاغ عن المخالفة، whistleblower، reward، report violation',
    'utilities_electricity':              'كهرباء، فاتورة كهرباء، من يدفع الكهرباء، electricity، electricity bill، utility',
    'utilities_water':                    'مياه، فاتورة مياه، من يدفع المياه، water، water bill، utility',
    'utilities_gas':                      'غاز، فاتورة غاز، من يدفع الغاز، gas، gas bill، utility',
    'parking_fee':                        'موقف سيارة، مواقف، أجرة موقف، parking، parking fee، parking spot، car park',
    'brokerage_fee':                      'سعي، عمولة وسيط، 2.5%، رسوم الوسيط، brokerage، commission، agent fee، broker fee',
    'rent_payment_cycle':                 'دورة سداد، شهري، ربع سنوي، سنوي، payment cycle، monthly، quarterly، annual payment',
    'permitted_use':                      'استخدام العقار، غرض سكني، غرض تجاري، permitted use، residential، commercial، allowed use',
    'common_areas_care':                  'أجزاء مشتركة، مشترك، مناطق مشتركة، common areas، shared spaces، shared facilities',
    'no_obstruct_maintenance':            'منع الصيانة، ترميم، إذن الصيانة، obstruct maintenance، access for repair، maintenance access',
    'handover_condition':                 'تسليم العقار، نموذج تسليم، إخلاء وتسليم، حالة التسليم، handover، property handover، move out',
    'daily_rent_overstay':                'غرامة تأخر إخلاء، أجرة يومية، التأخر في الإخلاء، overstay، holdover، daily penalty',
    'landlord_maintenance_duty':          'صيانة دورية، التزام المؤجر، مسؤولية المؤجر، landlord maintenance، periodic maintenance',
    'landlord_structural_repair':         'صيانة هيكلية، إصلاح عطل، سلامة المبنى، structural repair، landlord obligation، building repair',
    'tenant_minor_repairs':               'صيانة معتادة، مستأجر، صيانة بسيطة، minor repairs، tenant obligation، routine maintenance',
    'ownership_transfer_protection':      'انتقال ملكية، مالك جديد، بيع العقار، حماية المستأجر، ownership transfer، sale of property، tenant rights',
    'contract_termination_grounds':       'فسخ العقد، إخلال، إنذار، 15 يوماً، termination، breach، grounds for termination',
    'eviction_illegal_use':               'استخدام غير مشروع، إخلاء، مخالفة، آداب عامة، illegal use، eviction، prohibited use',
    'eviction_dangerous_alterations':     'تغييرات خطرة، إخلاء، سلامة العقار، dangerous alterations، eviction، unsafe modifications',
    'dispute_resolution_timeline':        'حل نزاع، ودي، 15 يومًا، تسوية ودية، dispute resolution، settlement، amicable resolution',
    'dispute_costs':                      'تكاليف نزاع، مماطلة، أتعاب محاماة، litigation costs، legal fees، dispute expenses',
    'automatic_rent_increase':            'زيادة تلقائية، نسبة زيادة، رفع الإيجار، مخالف، automatic increase، escalation clause، illegal increase',
    'immediate_eviction_clause':          'إخلاء فوري، بدون إشعار، طرد تعسفي، مخالف، immediate eviction، illegal eviction، no notice',
    'full_maintenance_on_tenant':         'صيانة كاملة على المستأجر، استغلال، مخالف، unfair maintenance، tenant overburden، illegal clause',
    'renewal_with_higher_rent':           'تجديد بأجرة أعلى، زيادة عند التجديد، مخالف، renewal increase، illegal renewal، rent hike',
    'eviction_nonpayment_30days':         'إخلاء عدم سداد، 30 يوماً، مهلة السداد، eviction notice، non-payment، 30 day notice',
    'eviction_structural_defect_report':  'عيوب هيكلية، تقرير فني، جهة حكومية، structural report، building defect، safety report',
    'eviction_illegal_activity':          'نشاط غير مشروع، آداب عامة، مخالفة، illegal activity، eviction، prohibited activity',
    'penalty_rent_increase_first':        'غرامة زيادة إيجار، مرة أولى، شهرين، first offense، fine، penalty، عقوبة',
    'penalty_rent_increase_second':       'غرامة زيادة إيجار، مرة ثانية، 6 أشهر، second offense، fine، penalty، عقوبة',
    'penalty_rent_increase_third':        'غرامة زيادة إيجار، مرة ثالثة، 12 شهراً، third offense، fine، penalty، عقوبة',
    'maintenance_landlord_periodic':      'صيانة دورية، مؤجر، نفقات صيانة، periodic maintenance، landlord duty، regular maintenance',
    'maintenance_landlord_structural':    'صيانة هيكلية، مؤجر، سلامة المبنى، structural maintenance، landlord repair، building safety',
    'maintenance_tenant_ordinary':        'صيانة معتادة، مستأجر، تكاليف صيانة، routine maintenance، tenant responsibility، minor upkeep',
    'rent_increase_exception_renovation': 'استثناء زيادة، ترميم، اعتراض على السقف، renovation exception، rent objection، landlord appeal',
    'rent_increase_exception_pre2024':    'استثناء عقد قبل 2024، اعتراض على السقف، pre-2024 lease، rent objection، old contract',
    'termination_property_collapse':      'سقوط العقار، تقرير فني، انقضاء العقد، 30 يوماً، property collapse، building collapse، force majeure',
    'termination_government_decision':    'قرار حكومي، تعديل أنظمة، انقضاء العقد، government decision، regulatory change، termination',
    'termination_state_expropriation':    'تملك الدولة، نزع ملكية، انقضاء العقد، expropriation، government acquisition، compulsory purchase',
    'termination_force_majeure':          'قوة قاهرة، ظروف استثنائية، انقضاء العقد، force majeure، act of god، extraordinary circumstances',
    'service_fees_electricity':           'من يدفع الكهرباء، فاتورة كهرباء، مسؤولية الدفع، electricity bill، who pays electricity، utility responsibility',
    'service_fees_water':                 'من يدفع المياه، فاتورة مياه، مسؤولية الدفع، water bill، who pays water، utility responsibility',
    'service_fees_gas':                   'من يدفع الغاز، فاتورة غاز، مسؤولية الدفع، gas bill، who pays gas، utility responsibility',
    'property_fitness_for_use':           'تسليم صالح، حالة العقار، ملحقات، fit for use، habitable، property condition، ready to use',
    'landlord_withhold_keys':             'حبس المفاتيح، أجرة معجلة، تسليم المفاتيح، withhold keys، advance rent، key handover',
    'hidden_defects_warranty':            'عيوب خفية، ضمان، عيوب غير ظاهرة، hidden defects، latent defects، warranty، concealed defects',
    'hidden_defect_remedy':               'عيب خفي، فسخ، إنقاص أجرة، تعويض عيوب، defect remedy، rent reduction، compensation for defects',
    'void_defect_exemption':              'بطلان شرط إعفاء، شرط باطل، إعفاء من الضمان، void clause، defect waiver، null clause',
    'no_landlord_interference':           'حظر تعرض المؤجر، انتفاع المستأجر، quiet enjoyment، landlord interference، peaceful possession',
    'third_party_interference_warranty':  'ضمان تعرض الغير، سبب نظامي، فسخ، third party interference، legal disturbance، warranty of possession',
    'emergency_termination_compensation': 'فسخ طارئ، تعويض، عذر طارئ، emergency termination، compensation، unexpected termination',
    'rent_cap_ownership_transfer':        'انتقال ملكية، حد سعري، مالك جديد، بيع العقار، ownership transfer، rent cap، new owner',
    'fixed_improvements_prohibition':     'تحسينات ثابتة، ديكور، إزالة تحسينات، fixed improvements، permanent fixtures، no removal',
    'rent_payment_obligation_429':        'سداد الأجرة، مواعيد السداد، إخلال بالسداد، payment obligation، rent due، failure to pay rent',
    'tenant_care_obligation_430':         'محافظة على العقار، تعدي، إهمال، tenant care، property care، damage، negligence',
}

for field, new_kw in keywords_map.items():
    df.loc[df['field'] == field, 'keywords'] = new_kw

df.to_excel("rental_contracts_master_clean.xlsx", index=False)
print(f"تم تحديث الـ keywords ✅  ({len(keywords_map)} بند)")

تم تحديث الـ keywords ✅  (76 بند)


In [6]:
# ── Step 7: تحقق نهائي ──
df = pd.read_excel("rental_contracts_master_clean.xlsx")

print(f"البنود:    {len(df)}")
print(f"nulls:     {df.isnull().sum().sum()}")
print(f"مكررات:   {df['field'].duplicated().sum()}")

# keywords — كل بند فيه إنجليزي؟
no_en = [r['field'] for _, r in df.iterrows()
         if not any(c.isascii() and c.isalpha() for c in str(r['keywords']))]
print(f"بدون keywords إنجليزي: {no_en if no_en else 'لا يوجد ✅'}")

# توزيع المراجع
sources = df['reference'].str.extract(r'^([^—]+)')[0].str.strip().value_counts()
print("\nتوزيع المراجع:")
for src, cnt in sources.items():
    print(f"  {cnt:>2}  {src}")

print("\n✅ الملف جاهز للرفع على GitHub")

البنود:    76
nulls:     0
مكررات:   0
بدون keywords إنجليزي: لا يوجد ✅

توزيع المراجع:
  44  نموذج العقد الإيجاري الموحد (منصة إيجار)
  17  الأحكام النظامية لضبط العلاقة الإيجارية (م/73 لعام 1447هـ)
  10  نظام المعاملات المدنية (م/191 لعام 1444هـ)
   4  نظام الوساطة العقارية (م/130 لعام 1443هـ)
   1  قرار هيئة العقار (سبتمبر 2025م)

✅ الملف جاهز للرفع على GitHub
